# LiveCodeBench data quick peek

Load the original LiveCodeBench dataset and inspect columns plus a sample row.


In [2]:
from datasets import load_dataset

DATASET_NAME = "livecodebench/execution-v2"

ds = load_dataset(DATASET_NAME)
ds


DatasetDict({
    test: Dataset({
        features: ['question_id', 'id', 'function_name', 'code', 'input', 'output', 'numsteps', 'problem_id', 'contest_id', 'contest_date', 'difficulty'],
        num_rows: 479
    })
})

In [3]:
# Pick a split to inspect (prefer train if available)
if isinstance(ds, dict):
    split_name = "train" if "train" in ds else list(ds.keys())[0]
    split = ds[split_name]
else:
    split_name = "(default)"
    split = ds

split_name, split


('test',
 Dataset({
     features: ['question_id', 'id', 'function_name', 'code', 'input', 'output', 'numsteps', 'problem_id', 'contest_id', 'contest_date', 'difficulty'],
     num_rows: 479
 }))

In [4]:
# Columns / keys
split.column_names


['question_id',
 'id',
 'function_name',
 'code',
 'input',
 'output',
 'numsteps',
 'problem_id',
 'contest_id',
 'contest_date',
 'difficulty']

In [5]:
# Sample row
split[0]


{'question_id': 2777,
 'id': 'sample_0',
 'function_name': 'distinctDifferenceArray',
 'code': 'def distinctDifferenceArray(a: List[int]) -> List[int]:\n    return [len(set(a[:i+1]))-len(set(a[i+1:]))for i in range(len(a))]',
 'input': 'distinctDifferenceArray(a = [1, 2, 3, 4, 5])',
 'output': '[-3, -1, 1, 3, 5]',
 'numsteps': 678,
 'problem_id': [0, 2, 0],
 'contest_id': 'weekly-contest-344',
 'contest_date': datetime.datetime(2023, 5, 7, 0, 0),
 'difficulty': 'easy'}

## Near-duplicate code by normalized AST

This normalizes identifiers (variable/function names) and groups code by structure.
Adjust `MAX_ROWS` if you want to scan more or less of the dataset.


In [6]:
import ast
from collections import defaultdict

class _NormalizeIdentifiers(ast.NodeTransformer):
    def __init__(self):
        self.name_map = {}
        self.counter = 0

    def _get(self, name):
        if name not in self.name_map:
            self.counter += 1
            self.name_map[name] = f"v{self.counter}"
        return self.name_map[name]

    def visit_FunctionDef(self, node):
        node = self.generic_visit(node)
        node.name = "func"
        return node

    def visit_AsyncFunctionDef(self, node):
        node = self.generic_visit(node)
        node.name = "func"
        return node

    def visit_ClassDef(self, node):
        node = self.generic_visit(node)
        node.name = "Class"
        return node

    def visit_arg(self, node):
        node.arg = self._get(node.arg)
        return node

    def visit_Name(self, node):
        return ast.copy_location(ast.Name(id=self._get(node.id), ctx=node.ctx), node)

    def visit_Attribute(self, node):
        node = self.generic_visit(node)
        return node

    def visit_alias(self, node):
        node.name = self._get(node.name)
        if node.asname:
            node.asname = self._get(node.asname)
        return node

    def visit_ExceptHandler(self, node):
        node = self.generic_visit(node)
        if node.name:
            node.name = self._get(node.name)
        return node

    def visit_Global(self, node):
        node.names = [self._get(n) for n in node.names]
        return node

    def visit_Nonlocal(self, node):
        node.names = [self._get(n) for n in node.names]
        return node

def normalize_code(code_str):
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return None
    tree = _NormalizeIdentifiers().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.dump(tree, include_attributes=False)

MAX_ROWS = 20000
n_rows = min(MAX_ROWS, len(split))
subset = split.select(range(n_rows))
codes = subset["code"]

sig_to_rows = defaultdict(list)
bad_parse = 0
for i, code in enumerate(codes):
    sig = normalize_code(code)
    if sig is None:
        bad_parse += 1
        continue
    sig_to_rows[sig].append(i)

dup_groups = [idxs for idxs in sig_to_rows.values() if len(idxs) > 1]
dup_groups = sorted(dup_groups, key=len, reverse=True)

{
    "rows_scanned": n_rows,
    "bad_parse": bad_parse,
    "num_duplicate_groups": len(dup_groups),
    "largest_group_size": len(dup_groups[0]) if dup_groups else 0,
}


{'rows_scanned': 479,
 'bad_parse': 0,
 'num_duplicate_groups': 20,
 'largest_group_size': 4}

In [7]:
# Inspect all duplicate groups
for group in dup_groups:
    print(f"=== Group of size {len(group)} ===")
    for idx in group:
        print(f"--- Row {idx} ---")
        print(codes[idx])
    print()


=== Group of size 4 ===
--- Row 380 ---
def countPairs(nums: List[int], target: int) -> int:
    n = len(nums)
    ans = 0
    
    for i in range(n):
        for j in range(i+1,n):
            if nums[i]+nums[j]<target:
                ans+=1
    
    return ans
--- Row 381 ---
def countPairs(nums: List[int], t: int) -> int:
    n=len(nums)
    res=0
    for i in range(n):
        for j in range(i+1,n):
            if nums[i]+nums[j]<t:
                res+=1
    return res
--- Row 384 ---
def countPairs(nums: List[int], target: int) -> int:
    n = len(nums)
    res = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + nums[j] < target:
                res += 1
    return res
--- Row 385 ---
def countPairs(nums: List[int], target: int) -> int:
    n = len(nums)
    ans = 0
    for i in range(n):
        for j in range(i + 1,n):
            if nums[i] + nums[j] < target:
                ans += 1
    return ans

=== Group of size 3 ===
--- Row 0 ---
def 

In [8]:

# Total Samples with Duplicates
total_dup_samples = sum(len(group) for group in dup_groups)
total_dup_samples

45

## Near matches via SimHash on AST features

This creates a coarse fingerprint from AST node types + normalized call names, then groups rows with
similar fingerprints (approximate near-duplicates). Increase `MAX_ROWS_NEAR` gradually.


In [9]:
import hashlib
from collections import defaultdict

def _hash64(s):
    return int(hashlib.md5(s.encode('utf-8')).hexdigest(), 16) & ((1 << 64) - 1)

def _simhash(features):
    # Basic 64-bit simhash
    v = [0] * 64
    for feat in features:
        h = _hash64(feat)
        for i in range(64):
            bit = (h >> i) & 1
            v[i] += 1 if bit else -1
    out = 0
    for i, val in enumerate(v):
        if val >= 0:
            out |= (1 << i)
    return out

def _hamming(a, b):
    return (a ^ b).bit_count()

def code_features(code_str):
    # Node types and call names with identifiers normalized
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return None
    tree = _NormalizeIdentifiers().visit(tree)
    ast.fix_missing_locations(tree)

    feats = []
    for node in ast.walk(tree):
        feats.append(type(node).__name__)
        if isinstance(node, ast.Call):
            # Capture called function 'shape'
            if isinstance(node.func, ast.Name):
                feats.append(f"call:{node.func.id}")
            elif isinstance(node.func, ast.Attribute):
                feats.append(f"call_attr:{node.func.attr}")
            else:
                feats.append("call:other")
    return feats

MAX_ROWS_NEAR = 20000
n_rows = min(MAX_ROWS_NEAR, len(split))
subset = split.select(range(n_rows))
codes = subset["code"]

hashes = []
bad_parse = 0
for code in codes:
    feats = code_features(code)
    if feats is None:
        hashes.append(None)
        bad_parse += 1
        continue
    hashes.append(_simhash(feats))

# LSH-style banding: split 64-bit hash into 8 bands of 8 bits
bands = 8
band_bits = 64 // bands
buckets = [defaultdict(list) for _ in range(bands)]
for idx, h in enumerate(hashes):
    if h is None:
        continue
    for b in range(bands):
        shift = b * band_bits
        key = (h >> shift) & ((1 << band_bits) - 1)
        buckets[b][key].append(idx)

# Candidate pairs from shared buckets
candidates = set()
for b in range(bands):
    for ids in buckets[b].values():
        if len(ids) < 2:
            continue
        ids = sorted(ids)
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                candidates.add((ids[i], ids[j]))

# Filter by Hamming distance
MAX_HAMMING = 6  # lower is stricter
near_pairs = []
for i, j in candidates:
    if hashes[i] is None or hashes[j] is None:
        continue
    if _hamming(hashes[i], hashes[j]) <= MAX_HAMMING:
        near_pairs.append((i, j))

{
    "rows_scanned": n_rows,
    "bad_parse": bad_parse,
    "candidate_pairs": len(candidates),
    "near_pairs": len(near_pairs),
}


{'rows_scanned': 479,
 'bad_parse': 0,
 'candidate_pairs': 101394,
 'near_pairs': 17162}

In [10]:
# Inspect a few near-match pairs
near_pairs[:5]


[(95, 386), (53, 160), (144, 278), (295, 447), (247, 376)]

In [11]:
# Show code for the first near-match pair
if near_pairs:
    i, j = near_pairs[0]
    print("A")
    print("-" * 80)
    print(subset[i]["code"])
    print("\nB")
    print("-" * 80)
    print(subset[j]["code"])
else:
    print("No near matches found with current settings.")


A
--------------------------------------------------------------------------------
def continuousSubarrays(nums: List[int]) -> int:
    l, r = 0, 0
    n = len(nums)
    cnt = Counter()
    ans = 0
    while l < n:
        while r < n and (len(cnt) == 0 or (nums[r] - min(cnt) <= 2 and max(cnt) - nums[r] <= 2)):
            cnt[nums[r]] += 1
            r += 1
        ans += r - l
        cnt[nums[l]] -= 1
        if cnt[nums[l]] == 0: del cnt[nums[l]]
        l += 1
    return ans

B
--------------------------------------------------------------------------------
def canMakeSubsequence(str1: str, str2: str) -> bool:
    n1, n2 = len(str1), len(str2)
    j = 0
    for i in range(n1):
        if str2[j] == 'a' and str1[i] == 'z':
            j += 1
        elif chr(ord(str2[j]) - 1) == str1[i] or str2[j] == str1[i]:
            j += 1
        if j == n2:
            return True
    return False


## Group near-duplicate pairs by function name

This uses the `near_pairs` results and groups them by shared `function_name`.


In [12]:
from collections import defaultdict

pair_groups = defaultdict(list)
missing_fn_in_pairs = 0

# Assumes `near_pairs` and `subset` are defined from the SimHash section
non_fn_match = 0
for i, j in near_pairs:
    name_i = subset[i].get("function_name")
    name_j = subset[j].get("function_name")
    if not name_i or not name_j:
        missing_fn_in_pairs += 1
        continue
    if name_i != name_j:
        non_fn_match += 1
        continue
    pair_groups[name_i].append((i, j))

{
    "near_pairs_total": len(near_pairs),
    "near_pairs_missing_function_name": missing_fn_in_pairs,
    "function_names_with_near_pairs": len(pair_groups),
    "near_pairs_different_function_names": non_fn_match,
}


{'near_pairs_total': 17162,
 'near_pairs_missing_function_name': 0,
 'function_names_with_near_pairs': 78,
 'near_pairs_different_function_names': 16568}

In [13]:
# Print a few near-duplicate groups for the same function_name
for name, pairs in list(sorted(pair_groups.items(), key=lambda x: len(x[1]), reverse=True))[:3]:
    print(f"{name}: {len(pairs)} pairs")
    print("examples", pairs[:5])


minOperations: 40 pairs
examples [(423, 425), (420, 422), (247, 425), (418, 425), (423, 426)]
maxSum: 28 pairs
examples [(161, 397), (158, 400), (398, 400), (160, 162), (161, 162)]
minimumSum: 21 pairs
examples [(171, 174), (255, 260), (256, 260), (257, 260), (171, 176)]


## Duplicate counts by `function_name` field

This ignores helpers and groups rows by the provided `function_name` column.


In [14]:
MAX_ROWS_FN_FIELD = 20000
n_rows = min(MAX_ROWS_FN_FIELD, len(split))
subset = split.select(range(n_rows))
fn_names = subset["function_name"]

fn_to_rows = defaultdict(list)
missing_fn = 0
for i, name in enumerate(fn_names):
    if name is None or (isinstance(name, str) and not name.strip()):
        missing_fn += 1
        continue
    fn_to_rows[name].append(i)

dup_groups = [idxs for idxs in fn_to_rows.values() if len(idxs) > 1]
dup_groups = sorted(dup_groups, key=len, reverse=True)

{
    "rows_scanned": n_rows,
    "missing_function_name": missing_fn,
    "num_duplicate_groups": len(dup_groups),
    "largest_group_size": len(dup_groups[0]) if dup_groups else 0,
}


{'rows_scanned': 479,
 'missing_function_name': 0,
 'num_duplicate_groups': 79,
 'largest_group_size': 20}

In [15]:
# Statistics on duplicate groups
group_sizes = [len(g) for g in dup_groups]
import numpy as np
np.mean(group_sizes), np.median(group_sizes), np.max(group_sizes)
total_dup_samples_fn_field = sum(group_sizes)
total_dup_samples_fn_field


473

In [16]:
total_non_duplicate_samples =  [idxs for idxs in fn_to_rows.values() if len(idxs) == 1]
len(total_non_duplicate_samples)


6

## Filter functions by non-boolean outputs

Collect `question_id` values for rows whose `output` is not a boolean.


In [17]:
def _annotation_to_name(node):
    if node is None:
        return None
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        return node.attr
    if isinstance(node, ast.Subscript):
        return _annotation_to_name(node.value)
    return None

def infer_bool_from_code(code_str, func_name):
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return None
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == func_name:
            ann = _annotation_to_name(node.returns)
            if ann is None:
                return None
            return ann.lower() == "bool"
    return None

def infer_bool_from_output(output_str):
    if output_str is None:
        return None
    if isinstance(output_str, str):
        if output_str == "True":
            return True
        if output_str == "False":
            return False
    return None

MAX_ROWS_OUTPUT = 20000
n_rows = min(MAX_ROWS_OUTPUT, len(split))
subset = split.select(range(n_rows))

non_bool_ids = set()
bool_ids = set()
missing_output = 0
missing_func = 0
unknown_type = 0

# Extra stats by question_id only for coverage tracking
non_bool_question_ids = set()
bool_question_ids = set()

for row in subset:
    row_id = row.get("id")
    qid = row.get("question_id")
    code = row.get("code")
    func_name = row.get("function_name")
    output_str = row.get("output")

    if not func_name:
        missing_func += 1
        continue

    is_bool = infer_bool_from_code(code, func_name)
    if is_bool is None:
        is_bool = infer_bool_from_output(output_str)

    if is_bool is None:
        unknown_type += 1
        continue

    if is_bool:
        bool_ids.add(row_id)
        bool_question_ids.add(qid)
    else:
        non_bool_ids.add(row_id)
        non_bool_question_ids.add(qid)

    if output_str is None:
        missing_output += 1

{
    "rows_scanned": n_rows,
    "missing_function_name": missing_func,
    "missing_output": missing_output,
    "unknown_type": unknown_type,
    "unique_bool_ids": len(bool_ids),
    "unique_non_bool_ids": len(non_bool_ids),
    "unique_bool_question_ids": len(bool_question_ids),
    "unique_non_bool_question_ids": len(non_bool_question_ids),
}


{'rows_scanned': 479,
 'missing_function_name': 0,
 'missing_output': 0,
 'unknown_type': 0,
 'unique_bool_ids': 47,
 'unique_non_bool_ids': 432,
 'unique_bool_question_ids': 8,
 'unique_non_bool_question_ids': 84}

## Filter functions with likely sentinel returns (heuristic)

This scans only `non_bool_ids` and flags functions whose return statements mix a sentinel with a non-sentinel.
If any function_name is flagged once, all rows with that function_name are filtered; mismatches are tracked.


In [18]:
SENTINEL_LITERALS = {None, "", -1}

def _const_value(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        if isinstance(node.operand, ast.Constant) and isinstance(node.operand.value, (int, float)):
            return -node.operand.value
    return None

def _is_sentinel(val):
    if val in SENTINEL_LITERALS:
        return True
    return False

def _expr_flags(node):
    """Return (has_sentinel, has_non_sentinel) for a return expression."""
    if isinstance(node, ast.IfExp):
        s1, n1 = _expr_flags(node.body)
        s2, n2 = _expr_flags(node.orelse)
        return (s1 or s2, n1 or n2)

    v = _const_value(node)
    if v is not None:
        if _is_sentinel(v):
            return (True, False)
        return (False, True)

    if isinstance(node, ast.List) and len(node.elts) == 0:
        return (True, False)
    if isinstance(node, ast.Tuple) and len(node.elts) == 0:
        return (True, False)
    if isinstance(node, ast.Set) and len(node.elts) == 0:
        return (True, False)
    if isinstance(node, ast.Dict) and len(node.keys) == 0:
        return (True, False)

    # Non-literal expression treated as non-sentinel.
    return (False, True)

def sentinel_heuristic(code_str, func_name):
    """Return True if function mixes sentinel with any other return value."""
    try:
        tree = ast.parse(code_str)
    except SyntaxError:
        return None

    returns = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == func_name:
            for sub in ast.walk(node):
                if isinstance(sub, ast.Return) and sub.value is not None:
                    returns.append(sub.value)

    if not returns:
        return None

    has_sentinel = False
    has_other = False
    for r in returns:
        s, n = _expr_flags(r)
        has_sentinel |= s
        has_other |= n

    return has_sentinel and has_other

# Only scan rows previously deemed non-boolean
MAX_ROWS_SENTINEL = 20000
n_rows = min(MAX_ROWS_SENTINEL, len(split))
subset = split.select(range(n_rows))

flagged_functions = {}  # name -> set(row ids)
clear_functions = {}    # name -> set(row ids)
missing_func = 0
unknown_flag = 0

for row in subset:
    row_id = row.get("id")
    if row_id not in non_bool_ids:
        continue
    func_name = row.get("function_name")
    if not func_name:
        missing_func += 1
        continue
    flag = sentinel_heuristic(row.get("code"), func_name)
    if flag is None:
        unknown_flag += 1
        continue
    if flag:
        flagged_functions.setdefault(func_name, set()).add(row_id)
    else:
        clear_functions.setdefault(func_name, set()).add(row_id)

# Final filter set: any function name flagged at least once
filtered_function_names = set(flagged_functions.keys())

# Track mismatches: functions flagged in some rows but not others
mismatch_report = {
    name: {
        "flagged_ids": sorted(flagged_functions.get(name, [])),
        "clear_ids": sorted(clear_functions.get(name, [])),
    }
    for name in filtered_function_names
    if clear_functions.get(name)
}

# Build final kept/filtered id sets for dataset generation
kept_ids = set()
filtered_out_ids = set()
missing_func_ids = set()
for row in subset:
    row_id = row.get("id")
    if row_id not in non_bool_ids:
        continue
    func_name = row.get("function_name")
    if not func_name:
        missing_func_ids.add(row_id)
        continue
    if func_name in filtered_function_names:
        filtered_out_ids.add(row_id)
    else:
        kept_ids.add(row_id)

{
    "rows_scanned": n_rows,
    "non_bool_ids_considered": len(non_bool_ids),
    "missing_function_name": missing_func,
    "unknown_flag": unknown_flag,
    "filtered_function_names": len(filtered_function_names),
    "mismatched_function_names": len(mismatch_report),
    "kept_ids": len(kept_ids),
    "filtered_out_ids": len(filtered_out_ids),
    "missing_func_ids": len(missing_func_ids),
}


{'rows_scanned': 479,
 'non_bool_ids_considered': 432,
 'missing_function_name': 0,
 'unknown_flag': 0,
 'filtered_function_names': 14,
 'mismatched_function_names': 9,
 'kept_ids': 331,
 'filtered_out_ids': 101,
 'missing_func_ids': 0}

In [19]:
# Show a few filtered functions and mismatches
list(sorted(filtered_function_names))[:10]


['alternatingSubarray',
 'findChampion',
 'findMinimumOperations',
 'lengthOfLongestSubsequence',
 'makeTheIntegerZero',
 'minGroupsForValidAssignment',
 'minOperations',
 'minSum',
 'minimumBeautifulSubstrings',
 'minimumIndex']

In [20]:
# Show mismatch details for a few function names
list(mismatch_report.items())[:10]


[('alternatingSubarray',
  {'flagged_ids': ['sample_351'],
   'clear_ids': ['sample_346',
    'sample_347',
    'sample_348',
    'sample_349',
    'sample_350']}),
 ('minGroupsForValidAssignment',
  {'flagged_ids': ['sample_266'],
   'clear_ids': ['sample_261',
    'sample_262',
    'sample_263',
    'sample_264',
    'sample_265']}),
 ('lengthOfLongestSubsequence',
  {'flagged_ids': ['sample_454', 'sample_456', 'sample_458'],
   'clear_ids': ['sample_455', 'sample_457', 'sample_459']}),
 ('minimumBeautifulSubstrings',
  {'flagged_ids': ['sample_358', 'sample_360'], 'clear_ids': ['sample_359']}),
 ('shortestBeautifulSubstring',
  {'flagged_ids': ['sample_249', 'sample_250', 'sample_251', 'sample_254'],
   'clear_ids': ['sample_252', 'sample_253']}),
 ('removeTrailingZeros',
  {'flagged_ids': ['sample_35'],
   'clear_ids': ['sample_30',
    'sample_31',
    'sample_32',
    'sample_33',
    'sample_34']}),
 ('minOperations',
  {'flagged_ids': ['sample_200',
    'sample_201',
    'sampl

In [21]:
# For all mismatch items, show the code for the flagged id vs the clear id
for name, info in list(mismatch_report.items())[:3]:
    flagged_ids = info["flagged_ids"]
    clear_ids = info["clear_ids"]
    print(f"=== Function: {name} ===")
    print(f"Flagged IDs: {flagged_ids}")
    for fid in flagged_ids:
        row = subset.filter(lambda x: x["id"] == fid)[0]
        print(f"\n--- Flagged ID {fid} ---")
        print(row["code"])
    print(f"\nClear IDs: {clear_ids}")
    for cid in clear_ids:
        row = subset.filter(lambda x: x["id"] == cid)[0]
        print(f"\n--- Clear ID {cid} ---")
        print(row["code"])
    print("\n\n")

=== Function: alternatingSubarray ===
Flagged IDs: ['sample_351']

--- Flagged ID sample_351 ---
def alternatingSubarray(nums: List[int]) -> int:
    res = 0
    for i in range(len(nums)):
        r = 1
        for j in range(i + 1, len(nums)):
            if nums[j] - nums[j - 1] == -1 + 2 * ((j - i) & 1):
                r += 1
                res = max(res, r)
            else:
                break
    return res if res > 0 else -1

Clear IDs: ['sample_346', 'sample_347', 'sample_348', 'sample_349', 'sample_350']

--- Clear ID sample_346 ---
def alternatingSubarray(nums: List[int]) -> int:
    n = len(nums)
    ans = -1
    for i in range(n):
        for j in range(i + 1, n):
            
            if nums[j] != nums[i] + ((j - i) & 1):
                break
            
            ans = max(ans, j - i + 1)
    return ans

--- Clear ID sample_347 ---
def alternatingSubarray(nums: List[int]) -> int:
    ans = -1
    n = len(nums)
    for i in range(n):
        delta = 1
        f

Filter: 100%|██████████| 479/479 [00:00<00:00, 26739.13 examples/s]



--- Flagged ID sample_266 ---
def minGroupsForValidAssignment(nums: List[int]) -> int:
    def count(unit):
        res = 0
        for value in counter.values():
            d, r = divmod(value, unit)
            if r > d:
                return -1
            res += -(-value // (unit + 1))
        return res
        
    counter, n = Counter(nums), len(nums)
    for unit in range(min(counter.values()), 0, -1):
        res = count(unit)
        if res != -1:
            return res

Clear IDs: ['sample_261', 'sample_262', 'sample_263', 'sample_264', 'sample_265']


Filter: 100%|██████████| 479/479 [00:00<00:00, 27876.28 examples/s]



--- Clear ID sample_261 ---
def minGroupsForValidAssignment(nums: List[int]) -> int:
    cnt = Counter(nums)
    freq = Counter(cnt.values())
    k = min(freq)
    ans = inf
    for i in range(1, k + 2):
        res = 0
        for x in freq:
            v = (x + i - 1) // i
            k1 = x - v * (i - 1)
            k2 = v - k1
            if k1 < 0 or k2 < 0: break
            res += freq[x] * v
        else: ans = min(ans, res)
    return ans


...


--- Clear ID sample_455 ---
def lengthOfLongestSubsequence(nums: List[int], target: int) -> int:
    nums.sort()
    dp = [0] * (target + 1)
    dp[0] = 1
    for x in nums:
        for i in range(target - x, -1, -1):
            if dp[i] > 0:
                dp[i+x] = max(dp[i+x], 1 + dp[i])
    return dp[-1] - 1


Filter: 100%|██████████| 479/479 [00:00<00:00, 27225.04 examples/s]



--- Clear ID sample_457 ---
def lengthOfLongestSubsequence(nums: List[int], target: int) -> int:
    max_len = [-1] * (target + 1)
    max_len[0] = 0
    for x in nums:
        for new_sum in reversed(range(x, target + 1)):
            if max_len[new_sum - x] != -1:
                max_len[new_sum] = max(
                    max_len[new_sum],
                    max_len[new_sum - x] + 1
                )
    return max_len[target]


Filter: 100%|██████████| 479/479 [00:00<00:00, 25744.79 examples/s]


--- Clear ID sample_459 ---
def lengthOfLongestSubsequence(nums: List[int], target: int) -> int:
    dp=[-1]*(target+1)
    dp[0]=0
    for a in nums:
        for i in range(target-a,-1,-1):
            if dp[i]==-1:continue
            dp[i+a]=max(dp[i+a],dp[i]+1)
    return dp[-1]





## Duplicates and difficulty on kept IDs

Recompute duplicate stats and difficulty distribution on the filtered `kept_ids` set.


In [22]:
# Build the kept subset
kept_rows = [row for row in subset if row.get("id") in kept_ids]
len(kept_rows)


331

In [23]:
# Exact AST duplicates on kept rows
kept_sig_to_rows = defaultdict(list)
bad_parse = 0
for i, row in enumerate(kept_rows):
    sig = normalize_code(row.get("code", ""))
    if sig is None:
        bad_parse += 1
        continue
    kept_sig_to_rows[sig].append(i)

kept_dup_groups = [idxs for idxs in kept_sig_to_rows.values() if len(idxs) > 1]
kept_dup_groups = sorted(kept_dup_groups, key=len, reverse=True)

{
    "kept_rows": len(kept_rows),
    "bad_parse": bad_parse,
    "num_duplicate_groups": len(kept_dup_groups),
    "largest_group_size": len(kept_dup_groups[0]) if kept_dup_groups else 0,
}


{'kept_rows': 331,
 'bad_parse': 0,
 'num_duplicate_groups': 17,
 'largest_group_size': 4}

In [24]:
total_dup_samples = sum(len(group) for group in kept_dup_groups)
total_dup_samples

38

In [25]:
# Duplicate groups by function_name on kept rows
kept_fn_to_rows = defaultdict(list)
missing_fn = 0
for i, row in enumerate(kept_rows):
    name = row.get("function_name")
    if not name:
        missing_fn += 1
        continue
    kept_fn_to_rows[name].append(i)

kept_fn_dup_groups = [idxs for idxs in kept_fn_to_rows.values() if len(idxs) > 1]
kept_fn_dup_groups = sorted(kept_fn_dup_groups, key=len, reverse=True)

{
    "kept_rows": len(kept_rows),
    "missing_function_name": missing_fn,
    "num_duplicate_groups": len(kept_fn_dup_groups),
    "largest_group_size": len(kept_fn_dup_groups[0]) if kept_fn_dup_groups else 0,
}


{'kept_rows': 331,
 'missing_function_name': 0,
 'num_duplicate_groups': 57,
 'largest_group_size': 12}

In [26]:
total_fn_dup_samples = sum(len(group) for group in kept_fn_dup_groups)
total_fn_dup_samples

325

In [27]:
# Difficulty distribution on kept rows
from collections import Counter

difficulty_counts = Counter()
missing_difficulty = 0
for row in kept_rows:
    diff = row.get("difficulty")
    if not diff:
        missing_difficulty += 1
        continue
    difficulty_counts[diff] += 1

{
    "difficulty_counts": dict(difficulty_counts),
    "missing_difficulty": missing_difficulty,
}


{'difficulty_counts': {'easy': 150, 'medium': 174, 'hard': 7},
 'missing_difficulty': 0}

In [28]:
# Save kept ids to a json
import json
kept_id_list = [row.get("id") for row in kept_rows]
with open("/work/pi_pgrabowicz_umass_edu/awyuan/task_based_compositional_generalization/data/livecodebench/livecodebench_filtered_ids.json", "w") as f:
    json.dump(kept_id_list, f, indent=2)

In [29]:
# Group kept ids by function_name, and then save to a json
fn_to_kept_ids = defaultdict(list)
for row in kept_rows:
    name = row.get("function_name")
    if not name:
        continue
    fn_to_kept_ids[name].append(row.get("id"))
with open("/work/pi_pgrabowicz_umass_edu/awyuan/task_based_compositional_generalization/data/livecodebench/livecodebench_filtered_ids_by_function.json", "w") as f:
    json.dump(fn_to_kept_ids, f, indent=2)


## Filter functions that error on random inputs

Use the same random-input logic as dataset generation to probe each kept function up to 3 times per row.
Filter out any function_name where error rate > 5% across all attempts in that group.


In [30]:
import ast
import importlib
import multiprocessing as mp
import random
import sys
from pathlib import Path
from collections import defaultdict

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
from init import read_config, ROOT_DIR
from scripts.generate_livecodebench import build_safe_builtins


cfg = read_config(f"{ROOT_DIR}/config/gen/livecodebench.yaml")
def parse_input_call(input_str):
    expr = ast.parse(input_str, mode="eval").body
    if not isinstance(expr, ast.Call):
        raise ValueError("Input is not a function call")
    if isinstance(expr.func, ast.Name):
        func_name = expr.func.id
    elif isinstance(expr.func, ast.Attribute):
        func_name = expr.func.attr
    else:
        raise ValueError("Unsupported function call")
    args = [ast.literal_eval(arg) for arg in expr.args]
    kwargs = {kw.arg: ast.literal_eval(kw.value) for kw in expr.keywords}
    return func_name, args, kwargs
def _bounded_int_range(value: int, cfg):
    scale = max(10, abs(value) * 2)
    low = -scale if value < 0 else 0
    high = scale
    if cfg.int_low is not None:
        low = max(low, cfg.int_low)
    if cfg.int_high is not None:
        high = min(high, cfg.int_high)
    if low > high:
        low, high = high, low
    return low, high
def _bounded_float_range(value: float, cfg):
    scale = max(10.0, abs(value) * 2.0)
    low = -scale if value < 0 else 0.0
    high = scale
    if cfg.float_low is not None:
        low = max(low, cfg.float_low)
    if cfg.float_high is not None:
        high = min(high, cfg.float_high)
    if low > high:
        low, high = high, low
    return low, high
def randomize_value(value, rng, cfg):
    if isinstance(value, bool):
        return rng.choice([True, False])
    if isinstance(value, int):
        low, high = _bounded_int_range(value, cfg)
        return rng.randint(low, high)
    if isinstance(value, float):
        low, high = _bounded_float_range(value, cfg)
        return rng.uniform(low, high)
    if isinstance(value, str):
        if not value:
            return value
        return "".join(rng.choice(cfg.string_alphabet) for _ in range(len(value)))
    if isinstance(value, list):
        return [randomize_value(v, rng, cfg) for v in value]
    if isinstance(value, tuple):
        return tuple(randomize_value(v, rng, cfg) for v in value)
    if isinstance(value, dict):
        return {k: randomize_value(v, rng, cfg) for k, v in value.items()}
    return value
def build_input_str(func_name, args, kwargs):
    args_str = ", ".join(repr(a) for a in args)
    kwargs_str = ", ".join(f"{k} = {repr(v)}" for k, v in kwargs.items())
    if args_str and kwargs_str:
        return f"{func_name}({args_str}, {kwargs_str})"
    if args_str:
        return f"{func_name}({args_str})"
    return f"{func_name}({kwargs_str})"
def _exec_worker(code, func_name, args, kwargs, result_q):
    try:
        exec_globals = {"__builtins__": build_safe_builtins()}
        exec_globals["typing"] = importlib.import_module("typing")
        exec_globals["List"] = exec_globals["typing"].List
        exec_globals["Dict"] = exec_globals["typing"].Dict
        exec_globals["Tuple"] = exec_globals["typing"].Tuple
        exec_globals["Set"] = exec_globals["typing"].Set
        exec_globals["Optional"] = exec_globals["typing"].Optional
        exec_globals["Deque"] = exec_globals["typing"].Deque
        exec_globals["DefaultDict"] = exec_globals["typing"].DefaultDict
        exec_globals["Counter"] = exec_globals["typing"].Counter
        exec_globals["Iterable"] = exec_globals["typing"].Iterable
        exec_globals["Iterator"] = exec_globals["typing"].Iterator
        exec_globals["Sequence"] = exec_globals["typing"].Sequence
        exec_globals["Mapping"] = exec_globals["typing"].Mapping
        exec_globals["MutableMapping"] = exec_globals["typing"].MutableMapping
        exec_globals["MutableSequence"] = exec_globals["typing"].MutableSequence
        exec_globals["MutableSet"] = exec_globals["typing"].MutableSet
        exec_globals["collections"] = importlib.import_module("collections")
        exec_globals["math"] = importlib.import_module("math")
        exec_globals["functools"] = importlib.import_module("functools")
        exec_globals["itertools"] = importlib.import_module("itertools")
        exec_globals["numpy"] = importlib.import_module("numpy")
        exec_globals["np"] = exec_globals["numpy"]
        exec_globals["heapq"] = importlib.import_module("heapq")
        exec_globals["bisect"] = importlib.import_module("bisect")
        exec_globals["operator"] = importlib.import_module("operator")
        exec_globals["deque"] = exec_globals["collections"].deque
        exec_globals["defaultdict"] = exec_globals["collections"].defaultdict
        exec_globals["Counter"] = exec_globals["collections"].Counter
        exec_globals["gcd"] = exec_globals["math"].gcd
        exec_globals["inf"] = exec_globals["math"].inf
        exec_globals["comb"] = exec_globals["math"].comb
        exec_globals["prod"] = exec_globals["math"].prod
        exec_globals["reduce"] = exec_globals["functools"].reduce
        exec_globals["cache"] = exec_globals["functools"].cache
        exec_globals["lru_cache"] = exec_globals["functools"].lru_cache
        exec_globals["accumulate"] = exec_globals["itertools"].accumulate
        exec_globals["islice"] = exec_globals["itertools"].islice
        exec_globals["combinations"] = exec_globals["itertools"].combinations
        exec_globals["heapify"] = exec_globals["heapq"].heapify
        exec_globals["heappush"] = exec_globals["heapq"].heappush
        exec_globals["heappop"] = exec_globals["heapq"].heappop
        exec(code, exec_globals, exec_globals)
        target = exec_globals.get(func_name)
        if target is None:
            solution_cls = exec_globals.get("Solution")
            if solution_cls is not None:
                target = getattr(solution_cls(), func_name, None)
        if target is None:
            raise ValueError(f"Function '{func_name}' not found")
        result = target(*args, **kwargs)
        result_q.put(("ok", result))
    except Exception as exc:
        result_q.put(("err", repr(exc)))
def run_with_timeout(code, func_name, args, kwargs, timeout):
    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx = mp.get_context("spawn")
    result_q = ctx.Queue()
    proc = ctx.Process(
        target=_exec_worker,
        args=(code, func_name, args, kwargs, result_q),
    )
    proc.start()
    proc.join(timeout)
    if proc.is_alive():
        proc.terminate()
        proc.join()
        return False, "Timeout"
    if result_q.empty():
        return False, "No result"
    status, payload = result_q.get()
    if status == "ok":
        return True, payload
    return False, payload
# Build lookup from row id -> row
id_to_row = {row.get("id"): row for row in kept_rows}
rng = random.Random(0)
MAX_TRIES_PER_ROW = 3
MAX_FN_ERROR_RATE = 0.05
fn_error_counts = defaultdict(int)
fn_attempt_counts = defaultdict(int)
parse_fail_ids = set()
for fn_name, id_list in fn_to_kept_ids.items():
    for row_id in id_list:
        row = id_to_row.get(row_id)
        if not row:
            continue
        input_str = row.get("input")
        code = row.get("code")
        try:
            _, base_args, base_kwargs = parse_input_call(input_str)
        except Exception:
            parse_fail_ids.add(row_id)
            continue
        for _ in range(MAX_TRIES_PER_ROW):
            args = [randomize_value(v, rng, cfg) for v in base_args]
            kwargs = {k: randomize_value(v, rng, cfg) for k, v in base_kwargs.items()}
            ok, _ = run_with_timeout(code, fn_name, args, kwargs, cfg.timeout_seconds)
            fn_attempt_counts[fn_name] += 1
            if not ok:
                fn_error_counts[fn_name] += 1
# Decide which functions to keep
filtered_function_names_random = set()
kept_function_names_random = set()
for fn_name, attempts in fn_attempt_counts.items():
    errors = fn_error_counts.get(fn_name, 0)
    if attempts == 0:
        continue
    if errors > MAX_FN_ERROR_RATE * attempts:
        filtered_function_names_random.add(fn_name)
    else:
        kept_function_names_random.add(fn_name)
# Build final filtered ids for dataset generation
random_filtered_out_ids = set()
random_kept_ids = set()
for fn_name, id_list in fn_to_kept_ids.items():
    if fn_name in filtered_function_names_random:
        random_filtered_out_ids.update(id_list)
    elif fn_name in kept_function_names_random:
        random_kept_ids.update(id_list)
{
    "function_names_total": len(fn_to_kept_ids),
    "function_names_attempted": len(fn_attempt_counts),
    "function_names_filtered": len(filtered_function_names_random),
    "function_names_kept": len(kept_function_names_random),
    "random_kept_ids": len(random_kept_ids),
    "random_filtered_out_ids": len(random_filtered_out_ids),
    "parse_fail_ids": len(parse_fail_ids),
}


{'function_names_total': 63,
 'function_names_attempted': 63,
 'function_names_filtered': 11,
 'function_names_kept': 52,
 'random_kept_ids': 269,
 'random_filtered_out_ids': 62,
 'parse_fail_ids': 0}

In [31]:
# Save filtered ids to json (overwrites previous outputs)
import json
random_kept_id_list = sorted(list(random_kept_ids))
with open("/work/pi_pgrabowicz_umass_edu/awyuan/task_based_compositional_generalization/data/livecodebench/livecodebench_filtered_ids.json", "w") as f:
    json.dump(random_kept_id_list, f, indent=2)

fn_to_random_kept_ids = {
    fn: [rid for rid in fn_to_kept_ids[fn] if rid in random_kept_ids]
    for fn in fn_to_kept_ids
    if fn in kept_function_names_random
}
with open("/work/pi_pgrabowicz_umass_edu/awyuan/task_based_compositional_generalization/data/livecodebench/livecodebench_filtered_ids_by_function.json", "w") as f:
    json.dump(fn_to_random_kept_ids, f, indent=2)


In [35]:
# Group fn_to_random_kept_ids by difficulty and size
from collections import Counter, defaultdict

# Build id -> row lookup (use kept_rows if present; fallback to subset)
id_to_row = {row.get("id"): row for row in kept_rows} if "kept_rows" in globals() else {row.get("id"): row for row in subset}

difficulty_order = {"easy": 0, "medium": 1, "hard": 2}

fn_stats = []
missing_difficulty_ids = set()
for fn, ids in fn_to_random_kept_ids.items():
    diffs = Counter()
    for rid in ids:
        row = id_to_row.get(rid)
        if not row:
            continue
        diff = row.get("difficulty")
        if not diff:
            missing_difficulty_ids.add(rid)
            continue
        diffs[diff] += 1
    total = sum(diffs.values())
    if not diffs:
        label = "unknown"
    elif len(diffs) == 1:
        label = next(iter(diffs))
    else:
        label = max(diffs, key=diffs.get)
    fn_stats.append({
        "function_name": fn,
        "count": len(ids),
        "difficulty_label": label,
        "difficulty_counts": dict(diffs),
    })

grouped = defaultdict(list)
for item in fn_stats:
    grouped[item["difficulty_label"]].append(item)

# Sort groups: easy -> medium -> hard -> unknown, then by count desc
ordered_groups = []
for label in ["easy", "medium", "hard", "unknown"]:
    items = sorted(grouped.get(label, []), key=lambda x: x["count"], reverse=True)
    ordered_groups.append((label, items))

ordered_groups, len(missing_difficulty_ids)


([('easy',
   [{'function_name': 'maxSum',
     'count': 12,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6, 'medium': 6}},
    {'function_name': 'countPairs',
     'count': 8,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6, 'medium': 2}},
    {'function_name': 'distinctDifferenceArray',
     'count': 6,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6}},
    {'function_name': 'minLength',
     'count': 6,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6}},
    {'function_name': 'makeSmallestPalindrome',
     'count': 6,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6}},
    {'function_name': 'distanceTraveled',
     'count': 6,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6}},
    {'function_name': 'longestAlternatingSubarray',
     'count': 6,
     'difficulty_label': 'easy',
     'difficulty_counts': {'easy': 6}},
    {'function_name': 'splitWordsBySep

Total:
269 Functions, 215 / 54 for 80/20 split

9 Holdout functions:
- Easy:
  1. minlength (6)
  2. makeSmallestPalindrome (6)
  3. distanceTraveled (6)
  4. longestAlternatingSubarray (6)
  5. splitWordsBySeparator (6)
- Medium:
  1. minimumCost (6)
  2. smallestString (6)
  3. maximumJumps (6)
  4. countCompleteSubarrays (6)


In [36]:
# Extract function names from the markdown list and print ids
import re

md_text = """Total:
  1. minlength (6)
  2. makeSmallestPalindrome (6)
  3. distanceTraveled (6)
  4. longestAlternatingSubarray (6)
  5. splitWordsBySeparator (6)
  6. minimumCost (6)
  7. smallestString (6)
  8. maximumJumps (6)
  9. countCompleteSubarrays (6)
"""
names = []
for line in md_text.splitlines():
    m = re.search(r"\d+\.\s*([A-Za-z_][A-Za-z0-9_]*)", line)
    if m:
        names.append(m.group(1))
    else:
        m2 = re.search(r"-\s*([A-Za-z_][A-Za-z0-9_]*)", line)
        if m2:
            names.append(m2.group(1))

# Deduplicate while preserving order
seen = set()
fn_list = []
for n in names:
    if n not in seen:
        seen.add(n)
        fn_list.append(n)

# Build case-insensitive lookup for keys in fn_to_random_kept_ids
lower_to_key = {k.lower(): k for k in fn_to_random_kept_ids.keys()}

for n in fn_list:
    if n in fn_to_random_kept_ids:
        ids = fn_to_random_kept_ids[n]
        print(f"{n}: {ids}")
        continue
    k = lower_to_key.get(n.lower())
    if k is None:
        print(f"{n}: [] (not found)")
        continue
    ids = fn_to_random_kept_ids[k]
    print(f"{k}: {ids}")


minLength: ['sample_18', 'sample_19', 'sample_20', 'sample_21', 'sample_22', 'sample_23']
makeSmallestPalindrome: ['sample_24', 'sample_25', 'sample_26', 'sample_27', 'sample_28', 'sample_29']
distanceTraveled: ['sample_57', 'sample_58', 'sample_59', 'sample_60', 'sample_61', 'sample_62']
longestAlternatingSubarray: ['sample_87', 'sample_88', 'sample_89', 'sample_90', 'sample_91', 'sample_92']
splitWordsBySeparator: ['sample_123', 'sample_124', 'sample_125', 'sample_126', 'sample_127', 'sample_128']
minimumCost: ['sample_36', 'sample_37', 'sample_38', 'sample_39', 'sample_40', 'sample_41']
smallestString: ['sample_50', 'sample_51', 'sample_52', 'sample_53', 'sample_54', 'sample_55']
maximumJumps: ['sample_97', 'sample_98', 'sample_99', 'sample_100', 'sample_101', 'sample_102']
countCompleteSubarrays: ['sample_140', 'sample_141', 'sample_142', 'sample_143', 'sample_144', 'sample_145']
